# Experiment 1: attractors and chaos

**Question.** Starting from random "soups" on a torus, how long does Conway's Game of Life wander before it falls into a cycle (the *transient*), what kind of cycle does it end on (the *period*), and how do both change with the grid size $L$?

Part 1a measures transients, part 1b the distribution of periods. Part 1c (damage spreading) follows later in this notebook.

## Predictions (written before any run)

| # | Question | Prediction |
|---|---|---|
| 1 | Typical transient on a 64 × 64 torus, density 0.35 | ~10 000 generations |
| 2 | Transient when $L$ doubles from 64 to 128 | grows like $L^2$ |
| 3 | Most common period; very long periods? | around 100; very long periods are rare |
| 6 | Any run not settling within 20 000 generations? | none |

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from life.metrics import detect_cycle
from life.rules import CONWAY
from life.topology import Lattice

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = ROOT / "docs" / "figures"
DATA = ROOT / "data" / "exp01"
FIGURES.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

DENSITY = 0.35
MAX_STEPS = 20_000
RUNS = {16: 200, 32: 200, 64: 200, 128: 50}
TORUS = Lattice("torus")

## 1a. Transient length and period

For each grid size $L$, draw random initial states (each cell alive with probability 0.35, seeds 0, 1, 2, …) and run `detect_cycle` with a cap of 20 000 generations. A run that hits the cap is recorded as *censored*, not dropped.

In [ ]:
records = []  # (L, seed, transient, period); -1 marks a censored run
for size, n_runs in RUNS.items():
    start = time.perf_counter()
    for seed in range(n_runs):
        soup = np.random.default_rng(seed).random((size, size)) < DENSITY
        cycle = detect_cycle(soup, CONWAY, TORUS, max_steps=MAX_STEPS)
        if cycle is None:
            records.append((size, seed, -1, -1))
        else:
            records.append((size, seed, cycle.transient, cycle.period))
    print(f"L = {size:>3}: {n_runs} runs in {time.perf_counter() - start:.1f} s")

results = np.array(records, dtype=np.int64)
np.savetxt(
    DATA / "cycles.csv",
    results,
    fmt="%d",
    delimiter=",",
    header="L,seed,transient,period",
    comments="",
)

In [ ]:
print(f"{'L':>4} {'runs':>5} {'censored':>9} {'median':>8} {'mean':>8} {'p90':>8} {'max':>8}")
for size in RUNS:
    rows = results[results[:, 0] == size]
    done = rows[rows[:, 2] >= 0, 2]
    print(
        f"{size:>4} {len(rows):>5} {np.sum(rows[:, 2] < 0):>9} {np.median(done):>8.0f} "
        f"{done.mean():>8.0f} {np.percentile(done, 90):>8.0f} {done.max():>8}"
    )

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

for size in RUNS:
    done = np.sort(results[(results[:, 0] == size) & (results[:, 2] >= 0), 2])
    left.step(done, np.arange(1, len(done) + 1) / len(done), where="post", label=f"L = {size}")
left.set_xscale("log")
left.set_xlabel("transient length (generations)")
left.set_ylabel("fraction of runs ≤ x")
left.set_title("Transient lengths (empirical CDF)")
left.legend()

sizes = np.array(list(RUNS))
medians = np.array(
    [np.median(results[(results[:, 0] == s) & (results[:, 2] >= 0), 2]) for s in sizes]
)
right.loglog(sizes, medians, "o-", label="median transient")
for power, style in [(1, ":"), (2, "--")]:
    right.loglog(
        sizes, medians[0] * (sizes / sizes[0]) ** power, style, color="grey", label=f"∝ L^{power}"
    )
right.set_xlabel("grid size L")
right.set_ylabel("generations")
right.set_title("Median transient vs. grid size")
right.legend()

fig.tight_layout()
fig.savefig(FIGURES / "exp01_transients.png", dpi=150)
plt.show()

## 1b. Which cycles do the soups end on?

In [ ]:
done = results[results[:, 2] >= 0]
periods, counts = np.unique(done[:, 3], return_counts=True)
print("period : runs (all sizes)")
for p, c in zip(periods, counts, strict=True):
    print(f"{p:>6} : {c}")

print()
for size in RUNS:
    rows = done[done[:, 0] == size]
    values, n = np.unique(rows[:, 3], return_counts=True)
    print(f"L = {size:>3}: " + ", ".join(f"{v}×{k}" for v, k in zip(values, n, strict=True)))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([str(p) for p in periods], counts, color="#2f7d4f")
ax.set_xlabel("period")
ax.set_ylabel("runs")
ax.set_title("Periods of the final cycles (all grid sizes)")
fig.tight_layout()
fig.savefig(FIGURES / "exp01_periods.png", dpi=150)
plt.show()

## Results vs. predictions (1a, 1b)

*To be written after looking at the output above.*